In [1]:
import os
from langchain_classic import hub
from langchain_groq import ChatGroq
from langchain_classic.agents import create_react_agent,Tool,AgentExecutor
from langchain_experimental.utilities import PythonREPL
from langchain_community.utilities import GoogleSerperAPIWrapper

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9952\1302743429.py:5: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL


In [2]:
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY=os.getenv("api_key")
SERPER_API_KEY=os.getenv("SERPER_API_KEY")



In [3]:
groq_llm=ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0


)



In [4]:
python_repl=PythonREPL()
# checling of the tools working correctly
python_repl.run(
    "print('hi , i am here to practice the agents ')"
)

repl_tool=Tool(
    name="python_tool",
    description=(
        "a pyhton shell to execute the python commands"
        "Input should be a valid python queries"
    ),
    func=python_repl.run
)

repl_tool.invoke(
    "print(3-7)"
)

Python REPL can execute arbitrary code. Use with caution.


'-4\n'

In [5]:
google_serper=GoogleSerperAPIWrapper()

google_serper.run("where is g7 summit in 2026 took place?")

serper_tool=Tool(
    name="Google",
    description=(
        "a tool used to serve on google " 
        "input should be a string"
    ),
    func=google_serper.run
)

serper_tool.invoke(
    "did pm Modi attended the g7 meeting in france?"
)


# Pulling the system prompt for the reAct agent
# Manually define the ReAct template
from langchain_classic.prompts import PromptTemplate
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Question: {input}
Thought: {agent_scratchpad}"""

react_prompt = PromptTemplate.from_template(template)

react_agent=create_react_agent(
    llm=groq_llm,
    tools=[repl_tool,serper_tool],
    prompt=react_prompt
)


In [6]:
agent_executer_one=AgentExecutor(
    agent=react_agent,
    tools=[repl_tool],
    verbose=True
)

user_input = (
    "If $ 450 amounts to $ 630 in 6 years, what will it amount to in 2 years "
    "at the same interest rate?"
)

response=agent_executer_one.invoke(
    {"input":user_input}
)

print(response)



> Entering new AgentExecutor chain...
Thought: To solve this problem, we need to first find the interest rate at which $450 amounts to $630 in 6 years. We can use the formula for compound interest: A = P(1 + r)^n, where A is the amount after n years, P is the principal amount, r is the interest rate, and n is the number of years.

Action: python_tool
Action Input: ```
import math
# Given values
P = 450
A = 630
n = 6
# Calculate the interest rate
r = (A/P)**(1/n) - 1
print(r)
```0.05768092640521627
Now that we have the interest rate, we can use it to find the amount after 2 years. We will use the same formula for compound interest: A = P(1 + r)^n, where A is the amount after n years, P is the principal amount, r is the interest rate, and n is the number of years.

Action: python_tool
Action Input: ```
import math
# Given values
P = 450
r = 0.05768092640521627
n = 2
# Calculate the amount after 2 years
A = P * (1 + r)**n
print(A)
```503.4100239366285
I now know the final answer

Final 

In [8]:
agent_executer_two=AgentExecutor(
    agent=react_agent,
    tools=[serper_tool,repl_tool],
    handle_parsing_errors=True,
    verbose=True
)

user_input="Find out what is the maximum speed of Vande Bharat train , and then calculate in how much time that train will travel to mumbai from delhi at that maximum speed ?"

response_two =agent_executer_two.invoke(
    {"input":user_input}
)

print(response_two["output"])



> Entering new AgentExecutor chain...
Thought: To find the maximum speed of the Vande Bharat train and calculate the time it takes to travel from Delhi to Mumbai, I first need to find the maximum speed of the train. 

Action: Google
Action Input: Vande Bharat train maximum speedIntroduced in 2018, the trainsets achieved speeds up to 183 km/h (114 mph) during trial runs. However, the maximum operational speed is restricted to 160 km/h ( ... Sleeper train achieved a remarkable speed of 180 kmph during its trial run in the Kota Division, marking a major milestone in Indian Railways' “ ... Most of the HDN and GD, GQ routes are fit for a maximum speed of 130KMPH. Track upgradation is needed to increase the speed. Most of the work ... Video · VANDE BHARAT EXPRESS running at Full Speed 155Kms/Hr #indiarailways #train #speed ... Indian Railways has successfully conducted a high-speed trial run of the Vande Bharat Sleeper train ahead of its full-scale rollout. Vande Bharat Express is India's 